In [ ]:
import pandas as pd
import numpy as np 

import sys
from langchain_community.vectorstores import FAISS
#from langchain.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings

#from langchain.embeddings import HuggingFaceEmbeddings




#from mistral import MistralClient
#from mistralai import Mistral
from mistralai.client import MistralClient



import requests



from datetime import datetime, timedelta
import math
from transformers import AutoTokenizer, AutoModel
import torch

import auth

e:\Apps\python 3.11\openagenda-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import mistralai
print(mistralai)
print(mistralai.__file__)

<module 'mistralai' from 'e:\\Apps\\python 3.11\\openagenda-env\\Lib\\site-packages\\mistralai\\__init__.py'>
e:\Apps\python 3.11\openagenda-env\Lib\site-packages\mistralai\__init__.py


In [6]:
import sys
print(sys.executable)

e:\Apps\python 3.11\openagenda-env\Scripts\python.exe


In [ ]:

url = "https://api.openagenda.com/v2/agendas/47052338/events"

today = datetime.today()
date_from = (today - timedelta(days=365)).strftime("%Y-%m-%d")
date_to = (today + timedelta(days=365)).strftime("%Y-%m-%d")

size = 100
page = 1
events_all = []

params = {
    "key": auths.TOKEN,
    "timings[from]": date_from,
    "timings[to]": date_to,
    "size": size,
    "page": page
}

response = requests.get(url, params=params)
data = response.json()

events_all.extend(data["events"])

# calcul du nombre total de pages
total = data["total"]
total_pages = math.ceil(total / size)

# récupérer les autres pages
for page in range(2, total_pages + 1):

    params["page"] = page
    response = requests.get(url, params=params)
    data = response.json()

    events_all.extend(data["events"])

# dataframe
events = data["events"]
df = pd.json_normalize(events_all)

#f["firstTiming.begin"] = pd.to_datetime(df["firstTiming.begin"])
#df["firstTiming.end"] = pd.to_datetime(df["firstTiming.end"])

print(df.head())
print(len(df))

   featured  attendanceMode imageCredits onlineAccessLink       uid  \
0     False               1         None             None  80262476   
1     False               1         None             None  56513581   
2     False               1         None             None  22276968   
3     False               1         None             None  88012351   
4     False               1         None             None     84445   

                            slug  status  \
0    hollie-cook-la-yegros-siska       1   
1          forever-pavot-yin-yin       1   
2  heavy-lungs-body-horror-chest       1   
3       blu-samu-ades-the-planet       1   
4     black-sea-dahu-1ere-partie       1   

                                    image.filename  image.size.width  \
0  0fb33e08983845debeb77b405973d8e4.base.image.jpg               700   
1  1f6b9ae6995e44b7a560d74159e0bd5e.base.image.jpg               700   
2  99b12b07d2c346beb79e829ec02c874b.base.image.jpg               700   
3  f3ccfe2ee97a4aabb

In [11]:
df["description.fr"].info()

<class 'pandas.Series'>
RangeIndex: 99 entries, 0 to 98
Series name: description.fr
Non-Null Count  Dtype
--------------  -----
99 non-null     str  
dtypes: str(1)
memory usage: 924.0 bytes


In [12]:
df

,featured,attendanceMode,imageCredits,onlineAccessLink,uid,slug,status,image.filename,image.size.width,image.size.height,...,firstTiming.begin,location.address,location.city,location.latitude,location.name,location.longitude,nextTiming.begin,nextTiming.end,dateRange.nl,nextTiming
0,False,1,None,None,80262476,hollie-cook-la-yegros-siska,1,0fb33e08983845debeb77b405973d8e4.base.image.jpg,700,700,...,2026-04-03T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,2026-04-03T20:00:00+02:00,2026-04-04T01:00:00+02:00,NaN,NaN
1,False,1,None,None,56513581,forever-pavot-yin-yin,1,1f6b9ae6995e44b7a560d74159e0bd5e.base.image.jpg,700,700,...,2026-04-09T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,2026-04-09T20:00:00+02:00,2026-04-10T01:00:00+02:00,NaN,NaN
2,False,1,None,None,22276968,heavy-lungs-body-horror-chest,1,99b12b07d2c346beb79e829ec02c874b.base.image.jpg,700,700,...,2026-04-11T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,2026-04-11T20:00:00+02:00,2026-04-12T01:00:00+02:00,NaN,NaN
3,False,1,None,None,88012351,blu-samu-ades-the-planet,1,f3ccfe2ee97a4aabbb8914cb21bdec8b.base.image.jpg,700,700,...,2026-04-24T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,2026-04-24T20:00:00+02:00,2026-04-25T01:00:00+02:00,NaN,NaN
4,False,1,None,None,84445,black-sea-dahu-1ere-partie,1,2aae0e2b5a044ef7bc0e339040878834.base.image.jpg,700,700,...,2026-04-30T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,2026-04-30T20:00:00+02:00,2026-05-01T01:00:00+02:00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,False,1,None,None,96654035,idles,1,fe9755ed27ba48e0aaefc2d6af8d293a.base.image.jpg,700,700,...,2025-06-04T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,NaN,NaN,NaN,NaN
95,False,1,None,None,86219516,youssoupha-joanne-radao-base,1,4e7648b57ed34bb192d09f7c0056a0b8.base.image.jpg,700,700,...,2025-05-23T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,NaN,NaN,NaN,NaN
96,False,1,None,None,42832059,coilguns-going-off-treaks,1,3d51ca0fe6a941ccb655b8015e51619e.base.image.jpg,700,700,...,2025-05-20T20:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,NaN,NaN,NaN,NaN
97,False,1,None,None,78028280,lodyssee-musicale-des-40-ans,1,f316fd9d45c347d4a747c39401bff7fb.base.image.jpg,700,700,...,2025-05-17T18:00:00.000+02:00,111 Boulevard Emile Delmas 17000 La Rochelle,La Rochelle,46.160216,La Sirène,-1.215559,NaN,NaN,NaN,NaN


In [13]:
df = pd.json_normalize(events)

df = df[[
    "uid",
    "title.fr",
    "description.fr",
    "location.name",
    "location.city",
    "firstTiming.begin",
    "firstTiming.end",
    "nextTiming.begin",
    "nextTiming.end",
    "lastTiming.begin"	
]]

print(df.head())

        uid                            title.fr     description.fr  \
0  80262476     HOLLIE COOK + LA YEGROS + SISKA        Dub, Reggae   
1  56513581             FOREVER PAVOT + YIN YIN  Pop, Psyché, Rock   
2  22276968  HEAVY LUNGS + BODY HORROR + CHEST.   Indie, Post-Punk   
3  88012351          BLU SAMU + ADÉS THE PLANET       Chanson, Rap   
4     84445              BLACK SEA DAHU + MEL D          Folk, Pop   

  location.name location.city              firstTiming.begin  \
0     La Sirène   La Rochelle  2026-04-03T20:00:00.000+02:00   
1     La Sirène   La Rochelle  2026-04-09T20:00:00.000+02:00   
2     La Sirène   La Rochelle  2026-04-11T20:00:00.000+02:00   
3     La Sirène   La Rochelle  2026-04-24T20:00:00.000+02:00   
4     La Sirène   La Rochelle  2026-04-30T20:00:00.000+02:00   

                 firstTiming.end           nextTiming.begin  \
0  2026-04-04T01:00:00.000+02:00  2026-04-03T20:00:00+02:00   
1  2026-04-10T01:00:00.000+02:00  2026-04-09T20:00:00+02:00   
2  20

In [14]:
df["firstTiming.begin"] = pd.to_datetime(df["firstTiming.begin"], utc=True)
df["firstTiming.end"] = pd.to_datetime(df["firstTiming.end"], utc=True)

In [15]:
df["nextTiming.begin"] = pd.to_datetime(df["nextTiming.begin"], utc=True)
df["nextTiming.end"] = pd.to_datetime(df["nextTiming.end"], utc=True)
df["lastTiming.begin"] = pd.to_datetime(df["nextTiming.begin"], utc=True)

In [16]:
df

,uid,title.fr,description.fr,location.name,location.city,firstTiming.begin,firstTiming.end,nextTiming.begin,nextTiming.end,lastTiming.begin
0,80262476,HOLLIE COOK + LA YEGROS + SISKA,"Dub, Reggae",La Sirène,La Rochelle,2026-04-03 18:00:00+00:00,2026-04-03 23:00:00+00:00,2026-04-03 18:00:00+00:00,2026-04-03 23:00:00+00:00,2026-04-03 18:00:00+00:00
1,56513581,FOREVER PAVOT + YIN YIN,"Pop, Psyché, Rock",La Sirène,La Rochelle,2026-04-09 18:00:00+00:00,2026-04-09 23:00:00+00:00,2026-04-09 18:00:00+00:00,2026-04-09 23:00:00+00:00,2026-04-09 18:00:00+00:00
2,22276968,HEAVY LUNGS + BODY HORROR + CHEST.,"Indie, Post-Punk",La Sirène,La Rochelle,2026-04-11 18:00:00+00:00,2026-04-11 23:00:00+00:00,2026-04-11 18:00:00+00:00,2026-04-11 23:00:00+00:00,2026-04-11 18:00:00+00:00
3,88012351,BLU SAMU + ADÉS THE PLANET,"Chanson, Rap",La Sirène,La Rochelle,2026-04-24 18:00:00+00:00,2026-04-24 23:00:00+00:00,2026-04-24 18:00:00+00:00,2026-04-24 23:00:00+00:00,2026-04-24 18:00:00+00:00
4,84445,BLACK SEA DAHU + MEL D,"Folk, Pop",La Sirène,La Rochelle,2026-04-30 18:00:00+00:00,2026-04-30 23:00:00+00:00,2026-04-30 18:00:00+00:00,2026-04-30 23:00:00+00:00,2026-04-30 18:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...
94,96654035,IDLES,"Post-Punk, Rock",La Sirène,La Rochelle,2025-06-04 18:00:00+00:00,2025-06-05 00:00:00+00:00,NaT,NaT,NaT
95,86219516,YOUSSOUPHA + JOANNE RADAO + BASE,"Chanson, Rap",La Sirène,La Rochelle,2025-05-23 18:00:00+00:00,2025-05-23 23:00:00+00:00,NaT,NaT,NaT
96,42832059,COILGUNS + GOING OFF + TREAKS,"Hardcore, Noise, Rock",La Sirène,La Rochelle,2025-05-20 18:00:00+00:00,2025-05-20 23:00:00+00:00,NaT,NaT,NaT
97,78028280,L'ODYSSÉE MUSICALE DES 40 ANS !,L'ASSOCIATION DE L'ÉCOLE DE MUSIQUE DE PÉRIGNY...,La Sirène,La Rochelle,2025-05-17 16:00:00+00:00,2025-05-17 21:59:00+00:00,NaT,NaT,NaT


In [17]:
descriptions = df["description.fr"].fillna("").tolist()

In [18]:
import accelerate

In [19]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.5.1+cu121
True
12.1


In [20]:
print(torch.cuda.is_available())  # True si GPU dispo
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(torch.cuda.current_device()))

True
0
NVIDIA GeForce RTX 4070 Ti SUPER


In [21]:
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU disponible : True
Nom du GPU : NVIDIA GeForce RTX 4070 Ti SUPER


In [22]:
print(torch.version.cuda)

12.1


In [24]:
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

In [28]:
from sentence_transformers import SentenceTransformer
#static-retrieval-mrl-en-v1
model = SentenceTransformer('all-mpnet-base-v2')

df['embeddings'] = list(model.encode(df['description.fr'].tolist(), convert_to_numpy=True))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11633.79it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
from datetime import datetime, timedelta, timezone

In [30]:
#
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

question = "pop"
question_vec = model.encode([question], convert_to_numpy=True)


#----------------------------------
#now = datetime.now()
now = datetime.now(timezone.utc)

question_lower = question.lower()

df_search = df

if "récemment" in question_lower or "recent" in question_lower:
    limit_date = now - timedelta(days=7)
    df_search = df[df["lastTiming.begin"] >= limit_date]

elif "aujourd" in question_lower:
    today = now.replace(hour=0, minute=0, second=0, microsecond=0)
    df_search = df[df["lastTiming.begin"] >= today]

elif "ce mois" in question_lower:
    limit_date = now - timedelta(days=30)
    df_search = df[df["lastTiming.begin"] >= limit_date]



# Construire une matrice d'embeddings
emb_matrix = np.stack(df['embeddings'].values)

# Calculer similarité cosinus
similarities = cosine_similarity(question_vec, emb_matrix)
best_idx = np.argmax(similarities)

# Récupérer toutes les infos
best_event = df.iloc[best_idx]

print("Description :", best_event['description.fr'])
print("Ville :", best_event['location.city'])
print("Lieu :", best_event['location.name'])
print("first Horaire debut :", best_event['firstTiming.begin'])
print("first Horaire fin :", best_event['firstTiming.end'])
print("last Horaire :", best_event['lastTiming.begin'])
print("prochain Horaire :", best_event['nextTiming.begin'])
print("prochain Horaire :", best_event['nextTiming.end'])
print("titre :", best_event['title.fr'])



Description : Pop
Ville : La Rochelle
Lieu : La Sirène
first Horaire debut : 2025-12-18 19:00:00+00:00
first Horaire fin : 2025-12-19 00:00:00+00:00
last Horaire : NaT
prochain Horaire : NaT
prochain Horaire : NaT
titre : DISIZ + BÉESAU


In [33]:
import faiss

emb_matrix = emb_matrix.astype("float32")

dim = emb_matrix.shape[1]


index = faiss.IndexFlatL2(dim)


index.add(emb_matrix)

In [34]:
question_vec = model.encode([question], convert_to_numpy=True)
question_vec = question_vec.astype("float32")


distances, indices = index.search(question_vec, 5)

In [35]:
for i, idx in enumerate(indices[0]):
    event = df.iloc[idx]
    
    print(f"\nRésultat {i+1}")
    print("Titre :", event['title.fr'])
    print("Ville :", event['location.city'])
    print("Description :", event['description.fr'])


Résultat 1
Titre : DISIZ + BÉESAU
Ville : La Rochelle
Description : Pop

Résultat 2
Titre : SHOWQUAI’S DE RENTRÉE DES STUDIOS
Ville : La Rochelle
Description : Pop, Électro

Résultat 3
Titre : MIKI + NEMONEMO
Ville : La Rochelle
Description : Chanson, Pop

Résultat 4
Titre : MALIK DJOUDI + FRÀNÇOIS AND THE ATLAS MOUNTAINS + GILDAA + CASSIEN
Ville : La Rochelle
Description : Chanson, Pop

Résultat 5
Titre : SUZANE + ANDREA PEMBAD'YS
Ville : La Rochelle
Description : Chanson, Pop


In [36]:

#apparemment normaliser

faiss.normalize_L2(emb_matrix)


faiss.normalize_L2(question_vec)


index = faiss.IndexFlatIP(dim)
index.add(emb_matrix)

In [37]:
df_search = df

if "récemment" in question_lower:
    limit_date = now - timedelta(days=7)
    df_search = df[df["lastTiming.begin"] >= limit_date]

emb_matrix = np.stack(df_search['embeddings'].values).astype("float32")

In [38]:
df_search = df_search.reset_index(drop=True)

In [39]:
faiss.write_index(index, "events.index")
index = faiss.read_index("events.index")

LANGCHAIN RAG

In [40]:
#pip install langchain langchain-community

In [41]:
from langchain.embeddings.base import Embeddings

class MyEmbedding(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True).tolist()

    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=True)[0].tolist()

In [42]:
def filter_docs(df, question):
    now = datetime.now(timezone.utc)
    question_lower = question.lower()

    df_search = df

    if "recent" in question_lower:
        limit_date = now - timedelta(days=7)
        df_search = df[df["lastTiming.begin"] >= limit_date]

    return df_search.reset_index(drop=True)

In [43]:
#vertorstore on recréé

df_filtered = filter_docs(df, question)

In [44]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    content = f"""
    Titre: {row['title.fr']}
    Description: {row['description.fr']}
    Ville: {row['location.city']}
    Lieu: {row['location.name']}
    """

    metadata = {
        "date": row["lastTiming.begin"],
        "city": row["location.city"]
    }

    documents.append(Document(page_content=content, metadata=metadata))

In [45]:
embedding_model = MyEmbedding(model)

vectorstore = FAISS.from_documents(documents, embedding_model)

In [46]:


retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [48]:
from datetime import datetime, timezone, timedelta
from sentence_transformers import SentenceTransformer
from langchain.embeddings.base import Embeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from transformers import pipeline
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


embedding_model = SentenceTransformer('all-mpnet-base-v2')

question = "événements pop seulement"

class MyEmbedding(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()







documents_all = []
for _, row in df.iterrows():
    content = f"""
    Titre: {row['title.fr']}
    Description: {row['description.fr']}
    Ville: {row['location.city']}
    Lieu: {row['location.name']}
    Date: {row['lastTiming.begin']}
    """
    documents_all.append(Document(
        page_content=content,
        metadata={"date": row["lastTiming.begin"], "city": row["location.city"]}
    ))




#  FAISS
vectorstore = FAISS.from_documents(documents_all, MyEmbedding(embedding_model))
retriever = vectorstore.as_retriever(search_kwargs={"k": 20})




pipe = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.1",
    max_new_tokens=300,
    temperature=0.7,
    device_map="auto"
)
llm = HuggingFacePipeline(pipeline=pipe)




prompt = PromptTemplate.from_template("""
Tu es un assistant qui recommande des événements.

Contexte:
{context}

Question:
{question}
donne moi des évenement de pop et pas de rock

Réponds en français avec une liste claire.
""")


chain = (
    prompt
    | llm
    | StrOutputParser()
)















def filter_docs(docs, question):
    now = datetime.now(timezone.utc)
    question_lower = question.lower()
    filtered = docs
    if "recent" in question_lower:
        filtered = [doc for doc in docs if doc.metadata["date"] >= now - timedelta(days=7)]
    return filtered














question = "événements pop seulement"

# FAISS
docs = retriever.invoke(question)

# "recent"
filtered_docs = filter_docs(docs, question)


context = "\n\n".join([doc.page_content for doc in filtered_docs])



#rep
response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11685.42it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 85.18it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the 


Tu es un assistant qui recommande des événements.

Contexte:

    Titre: DISIZ + BÉESAU
    Description: Pop
    Ville: La Rochelle
    Lieu: La Sirène
    Date: NaT
    


    Titre: ODEZENNE présente "GRAND PRIX SURPRISE-PARTIE"
    Description: Chanson, Electro, Hip-Hop
    Ville: La Rochelle
    Lieu: La Sirène
    Date: 2026-07-11 17:00:00+00:00
    


    Titre: SUZANE + ANDREA PEMBAD'YS
    Description: Chanson, Pop
    Ville: La Rochelle
    Lieu: La Sirène
    Date: NaT
    


    Titre: SHOWQUAI’S DE RENTRÉE DES STUDIOS
    Description: Pop, Électro
    Ville: La Rochelle
    Lieu: La Sirène
    Date: NaT
    


    Titre: KOMPROMAT + DOUBLE VITRAGE + NOMENKLATÜR + IPPON + TEDAAK + AFTER CLUB
    Description: Electro
    Ville: La Rochelle
    Lieu: La Sirène
    Date: NaT
    


    Titre: FFF + 1ÈRE PARTIE
    Description: Funk, Rock
    Ville: La Rochelle
    Lieu: La Sirène
    Date: 2026-10-02 18:00:00+00:00
    


    Titre: BLU SAMU + ADÉS THE PLANET
    Description: 

In [ ]:
df.to_csv("data.csv")